# DeepDiscover: groundwater-level observations

This notebook demonstrates calibration against groundwater head and streamflow. The physical model conserves water; the head operator converts one latent water storage into an observed head. These data are synthetic and are generated within the notebook.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from deepdiscover import (
    CRCNet, DiscoverUnit, MultiDUModel, DeepDiscover, HydroDataset,
    TrainableStorageToHead, PrescribedSink,
)

tf.keras.utils.set_random_seed(19)
rng = np.random.default_rng(19)
n = 420
t = np.arange(n)
p = rng.gamma(2, 1.5, n).astype('float32')
pet = (1.0 + 0.2 * np.sin(2*np.pi*t/180)).astype('float32')
pumping = np.zeros(n, 'float32')
pumping[200:260] = 0.25
s = 0.0
level, flow = [], []
for rain, extraction in zip(p, pumping):
    a = s + rain
    loss = min(extraction, a)
    b = a - loss
    q = 0.06*b
    s = b-q
    flow.append(q)
    level.append(105.0 + 0.035*s)
flow = np.asarray(flow, 'float32')
level = np.asarray(level, 'float32') + rng.normal(0, 0.03, n)
level[110:123] = np.nan

## Physical model and observation operators

Pumping is a prescribed physical sink. The head conversion has a positive learned coefficient and learned reference level.

In [ ]:
def build():
    du = DiscoverUnit(
        process_nets=[CRCNet([1, -1, 1], [5, 3, 100])],
        context_indices=[(0, 1)],
        exchanges=[PrescribedSink(driver_index=2)],
    )
    physical = MultiDUModel([du], [['outlet']])
    operators = {
        'flow': 'outlet',
        'groundwater_head': TrainableStorageToHead(du_index=0, scale_init=0.02, offset_init=100.0),
    }
    return DeepDiscover(physical, observations=operators)

dd = build()
data = HydroDataset(p[:, None], np.column_stack([p, pet, pumping]),
                    {'flow': flow, 'groundwater_head': level})
train = data.make_windows(warmup=25, score_length=65, stride=30, first_score=25, last_score=290)
validation = data.continuous_batch(290, 365, warmup=25)
test = data.continuous_batch(365, 420, warmup=25)
print(train.water.shape, train.masks['groundwater_head'].sum())

## Calibrate with two observations

The variance of each target is estimated from the scored training data. The objectives can also be weighted or replaced independently.

In [ ]:
history = dd.fit(train, validation, epochs=25, batch_size=4, learning_rate=0.003,
                 loss={'flow': 'mse', 'groundwater_head': 'huber'},
                 weights={'flow': 1.0, 'groundwater_head': 1.0},
                 normalize=True, patience=6, verbose=1)
print(dd.evaluate(test)['metrics'])

## Plot and save

Head observations are expressed in metres. The latent storage is expressed in water-equivalent millimetres. The affine conversion is an observation hypothesis.

In [ ]:
estimate = dd.predict(test.water, test.drivers)['groundwater_head'].numpy()[0,25:,0]
plt.figure(figsize=(10,3))
plt.plot(level[365:], label='Observed head')
plt.plot(estimate, label='DeepDiscover')
plt.ylabel('Hydraulic head (m)')
plt.xlabel('Test days')
plt.legend(); plt.tight_layout(); plt.show()

dd.save_checkpoint('results/groundwater_example', {'case': 'groundwater_head'})

In [ ]:
restored = build()
restored.predict(test.water, test.drivers)
restored.load_checkpoint('results/groundwater_example')
repeat = restored.predict(test.water, test.drivers)['groundwater_head'].numpy()
np.testing.assert_allclose(repeat, dd.predict(test.water, test.drivers)['groundwater_head'].numpy())
print('Checkpoint restored successfully')